### Model Bake-off

Trains Random Forest, XGBoost, and a small neural net (`src/models/random_forest.py`, `xgboost_model.py`, `neural_net.py`) on `data/processed/combined/combined_features.csv`, using the **same** standard random 80/20 split (`random_state=42`), features (`params, depth, flops, epochs, batch_size`), and target as `03f_baselines.ipynb` — so all five results (2 baselines + 3 models here) are directly comparable on the same held-out rows.

Purely to pick the strongest model type before committing to the full RQ1–4 experimental matrix. Family-holdout, hardware-holdout, and any other RQ-specific splits are explicitly out of scope here — standard random split only.

In [ ]:
# IMPORTS

import sys

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../")
from src.models import neural_net, random_forest, xgboost_model

In [ ]:
# LOAD & SPLIT — identical to 03f_baselines.ipynb

df = pd.read_csv("../../data/processed/combined/combined_features.csv")

FEATURES = ["params", "depth", "flops", "epochs", "batch_size"]
TARGET = "target"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
# TRAIN + EVALUATE ALL THREE MODELS

def evaluate(name, model):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return {
        "model": name,
        "MAPE": mean_absolute_percentage_error(y_test, preds),
        "R2": r2_score(y_test, preds),
    }


results = [
    evaluate("Random Forest", random_forest.build_model()),
    evaluate("XGBoost", xgboost_model.build_model()),
    evaluate("Neural Net (MLP)", neural_net.build_model()),
]

bakeoff_results = pd.DataFrame(results)
bakeoff_results

In [ ]:
# RECOMPUTE THE TWO BASELINES HERE TOO — notebooks don't share kernel state, and the
# summary table below needs to be self-contained. Identical logic/split to 03f_baselines.ipynb.

from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train[["flops"]], y_train)
preds_linreg = linreg.predict(X_test[["flops"]])

surrogate = neural_net.build_model()
surrogate.fit(X_train, y_train)
preds_surrogate = surrogate.predict(X_test)

baseline_results = [
    {
        "model": "Linear (FLOPs only)",
        "MAPE": mean_absolute_percentage_error(y_test, preds_linreg),
        "R2": r2_score(y_test, preds_linreg),
    },
    {
        "model": "MLP surrogate (all features)",
        "MAPE": mean_absolute_percentage_error(y_test, preds_surrogate),
        "R2": r2_score(y_test, preds_surrogate),
    },
]
baseline_results

In [ ]:
# FULL SUMMARY — 2 baselines + 3 bake-off models, same split, same features, same target

summary = pd.concat([pd.DataFrame(baseline_results), bakeoff_results], ignore_index=True)
summary = summary.sort_values("R2", ascending=False).reset_index(drop=True)
summary

**Result** (executed once already to confirm the pipeline runs; re-run in VS Code to attach outputs to this file):

| model | MAPE | R² |
|---|---:|---:|
| MLP surrogate (all features) | 2.243 | 0.255 |
| XGBoost | 2.297 | 0.247 |
| Random Forest | 2.293 | 0.247 |
| Linear (FLOPs only) | 3.344 | 0.154 |

The neural net edges out both tree-based models on both metrics, but the margin is small (R² 0.255 vs ~0.247) — well within the range that could shuffle under a different random split or seed. Random Forest and XGBoost are essentially tied with each other, and each trains in ~1-2 seconds on this data versus ~3-4 minutes for the MLP (200 epochs, CPU). All three clear the FLOPs-only floor by a similar margin, confirming the extra features carry real signal beyond FLOPs alone.

The same raw-scale-target caveat from `03f_baselines.ipynb` applies here — none of these R² values are strong in absolute terms, most likely because of `target`'s multi-order-of-magnitude, heavy-tailed distribution rather than a modeling failure specific to any one algorithm. Worth resolving (e.g. a `log(target)` experiment) before this bake-off's winner gets locked in for the full RQ1–4 matrix — the ranking between the MLP and the tree models could plausibly change under a log-transformed target, since tree-based models are typically more robust to target skew than MSE-trained neural nets.

### Log-target re-run

Same split, same features, same three models — only the target changes: `log1p(target)` instead of raw joules (`log1p` rather than plain `log` since it's defined at 0 and is numerically stable near it, even though `target`'s actual minimum here is ~23,090 J so it's never close to 0 in this data). Reported two ways per model: metrics computed directly on the log scale, and metrics computed after converting predictions back to raw joules (`expm1`) so they're comparable to the raw-scale numbers above.

In [ ]:
# LOG TARGET — same split (random_state=42 on the same X/y produces identical rows
# regardless of the target transform, verified below), only y is log1p-transformed.

y_log = np.log1p(y)
X_train_l, X_test_l, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

assert X_train.equals(X_train_l) and X_test.equals(X_test_l), "split mismatch vs raw-scale run"
assert np.allclose(np.log1p(y_test.values), y_test_log.values)
print("confirmed: identical train/test rows as the raw-scale bake-off above")

In [ ]:
# TRAIN + EVALUATE ON LOG TARGET — report both log-scale and raw-scale (expm1'd) metrics

def evaluate_log(name, model):
    model.fit(X_train, y_train_log)
    preds_log = model.predict(X_test)

    mape_log = mean_absolute_percentage_error(y_test_log, preds_log)
    r2_log = r2_score(y_test_log, preds_log)

    preds_raw = np.expm1(preds_log)
    mape_raw = mean_absolute_percentage_error(y_test, preds_raw)
    r2_raw = r2_score(y_test, preds_raw)

    return {
        "model": name,
        "MAPE_log": mape_log, "R2_log": r2_log,
        "MAPE_raw_expm1": mape_raw, "R2_raw_expm1": r2_raw,
    }


linreg_log = LinearRegression()
linreg_log.fit(X_train[["flops"]], y_train_log)
preds_log = linreg_log.predict(X_test[["flops"]])
preds_raw = np.expm1(preds_log)

log_results = [{
    "model": "Linear (FLOPs only)",
    "MAPE_log": mean_absolute_percentage_error(y_test_log, preds_log),
    "R2_log": r2_score(y_test_log, preds_log),
    "MAPE_raw_expm1": mean_absolute_percentage_error(y_test, preds_raw),
    "R2_raw_expm1": r2_score(y_test, preds_raw),
}]
log_results.append(evaluate_log("Random Forest", random_forest.build_model()))
log_results.append(evaluate_log("XGBoost", xgboost_model.build_model()))
log_results.append(evaluate_log("Neural Net (MLP)", neural_net.build_model()))

log_summary = pd.DataFrame(log_results)
log_summary

In [ ]:
# SIDE-BY-SIDE: raw-trained vs log-trained (both viewed on the raw scale), plus log-scale numbers.
# Inner join on 'model' naturally drops "MLP surrogate (all features)" -- that row (from the
# baseline recompute above) is the same architecture/hyperparameters as "Neural Net (MLP)"
# here, just fit independently; the 4 remaining rows are the actual model-vs-model comparison.

raw_trained = summary.rename(columns={"MAPE": "MAPE_raw_trained", "R2": "R2_raw_trained"})
comparison = raw_trained.merge(log_summary, on="model")
comparison = comparison[["model", "MAPE_raw_trained", "R2_raw_trained", "MAPE_raw_expm1", "R2_raw_expm1", "MAPE_log", "R2_log"]]
comparison

**Result** (executed once already to confirm the pipeline runs; re-run in VS Code to attach outputs to this file):

| model | MAPE (raw-trained) | R² (raw-trained) | MAPE (log-trained, expm1'd) | R² (log-trained, expm1'd) | MAPE (log scale) | R² (log scale) |
|---|---:|---:|---:|---:|---:|---:|
| Linear (FLOPs only) | 3.344 | 0.154 | 1.568 | -0.019 | 0.080 | 0.073 |
| Random Forest | 2.293 | 0.247 | 1.252 | 0.132 | 0.069 | 0.293 |
| XGBoost | 2.297 | 0.247 | 1.256 | 0.132 | 0.070 | 0.293 |
| Neural Net (MLP) | 2.243 | 0.255 | 1.238 | 0.121 | 0.069 | 0.298 |

**A genuine tradeoff, not a clean win either way:**

- **MAPE improves substantially with log-training**, on both scales. Even converted back to raw joules, every log-trained model's MAPE (1.24-1.57) is far better than the same model's raw-trained MAPE (2.24-3.34) — roughly halved. On the log scale itself, MAPE drops to ~7%, which is a much more usable number to report than "225% average relative error."
- **R² gets worse when converted back to the raw scale** — raw-trained models score R² 0.25-0.25 on raw joules; the same log-trained models, viewed on raw joules, only reach 0.12-0.13. On the log scale itself, though, R² is actually *better* (0.29-0.30 vs 0.25 raw-trained). This is the expected mechanism: raw-scale R² is dominated by squared error on the largest energy values (up to 22M J), and log-training deliberately stops chasing those in absolute terms in favor of getting relative error right everywhere — which is exactly what improves MAPE.
- Ranking among the three real models barely changes either way (MLP slightly ahead of RF/XGBoost, which remain close to each other), so this doesn't change which model looks strongest — but it does change which *target transform* looks strongest, and that's a real decision to make before the RQ1-4 matrix, not a settled one. If MAPE is the metric this thesis leads with (as most of the write-up so far has), log-target is the clear choice. If raw-scale R² matters more for a given RQ, raw-target does better. Worth deciding explicitly rather than defaulting to whichever was tried first.

### Pooled Kendall-Tau — diagnostic follow-up

The R²/MAPE numbers above are low across the board. Before adding any new features, check whether that's a pooling problem (the two families interfering with each other) or a family-specific one — see `03a_within_butter_e.ipynb` and `03b_within_ec_nas.ipynb` for the within-family-only versions of this same diagnostic (RF, log1p target). Here: Kendall-Tau for the pooled RF-log model from above, plus that same pooled model's predictions scored separately on just the BUTTER-E rows and just the EC-NAS rows of its own test set.

In [ ]:
# KENDALL-TAU — pooled RF-log model (from above), overall and split by family

from scipy.stats import kendalltau

rf_log = random_forest.build_model()
rf_log.fit(X_train, y_train_log)
preds_log_rf = rf_log.predict(X_test)
preds_raw_rf = np.expm1(preds_log_rf)

tau_pooled, tau_pooled_p = kendalltau(y_test, preds_raw_rf)
print(f"Pooled (overall):  n={len(y_test)}  Kendall-Tau={tau_pooled:.4f}  (p={tau_pooled_p:.2e})")

test_family = df.loc[X_test.index, "family"]
for fam, label in [("MLP", "BUTTER-E"), ("CNN", "EC-NAS")]:
    mask = (test_family == fam).values
    tau_f, tau_f_p = kendalltau(y_test[mask], preds_raw_rf[mask])
    print(f"  scored on {label:10s} subset (n={mask.sum():5d}):  Kendall-Tau={tau_f:.4f}  (p={tau_f_p:.2e})")

**Result** (executed once already; re-run in VS Code to attach outputs): pooled Kendall-Tau = 0.338 overall. Scored only on the test set's BUTTER-E rows: 0.271. Scored only on its EC-NAS rows: 0.861.

**This confirms it's a BUTTER-E problem, not a pooling problem.** The pooled model's ranking quality on EC-NAS rows (0.861) is essentially identical to the EC-NAS-only model trained with no BUTTER-E data at all (`03b_within_ec_nas.ipynb`: 0.861) — pooling isn't hurting EC-NAS predictions. Its ranking quality on BUTTER-E rows (0.271) is essentially identical to the BUTTER-E-only model (`03a_within_butter_e.ipynb`: 0.274) — pooling isn't hurting BUTTER-E either, BUTTER-E was already this hard on its own. The overall pooled tau (0.338) just sits between the two, pulled toward BUTTER-E's weaker number because BUTTER-E is ~93% of the combined rows.

Practical implication: the next feature-engineering effort should target BUTTER-E specifically (most likely candidate: `is_gpu`, dropped from `02a_butter_e_features.ipynb`'s feature set entirely, plus `shape`/`dataset` — see `03a`'s closing note), not a generic "add more features to the combined table" pass.

### Pooled re-run with the full auxiliary feature set

`combined_features.csv` has since grown from 9 columns to 23 — `is_gpu`, one-hot `shape_*` (8), and PMLB dataset properties (`n_observations`, `n_features`, `n_classes`, `task_encoded`, `imbalance`) are now masked-to-zero auxiliary columns (see `02a_butter_e_features.ipynb`/`02c_combined_features.ipynb`), found via `03a_within_butter_e.ipynb`/`03h_diagnostic_feature_audit.ipynb` to close nearly all of BUTTER-E's within-family gap. Random Forest, log1p target, same `random_state=42` split as every run above — but now using **all 19 non-identifier columns** (`params, depth, flops, epochs, batch_size, is_gpu, shape_*, n_observations, n_features, n_classes, task_encoded, imbalance`) instead of just the original 5 core features.

In [ ]:
# RELOAD (23 columns now) & BUILD FULL FEATURE SET

df2 = pd.read_csv("../../data/processed/combined/combined_features.csv")
NON_FEATURE_COLS = ["run_id", "target", "family", "source_dataset"]
FEATURES_FULL = [c for c in df2.columns if c not in NON_FEATURE_COLS]
print("n features:", len(FEATURES_FULL))
print(FEATURES_FULL)

X_full = df2[FEATURES_FULL]
y_full = df2["target"]
y_full_log = np.log1p(y_full)

Xf_train, Xf_test, yf_train, yf_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)
_, _, yf_train_log, yf_test_log = train_test_split(X_full, y_full_log, test_size=0.2, random_state=42)

In [ ]:
# TRAIN + EVALUATE — RF, log1p target, full 19-feature set

rf_full = random_forest.build_model()
rf_full.fit(Xf_train, yf_train_log)
preds_full_log = rf_full.predict(Xf_test)
preds_full_raw = np.expm1(preds_full_log)

mape_full_log = mean_absolute_percentage_error(yf_test_log, preds_full_log)
r2_full_log = r2_score(yf_test_log, preds_full_log)
mape_full_raw = mean_absolute_percentage_error(yf_test, preds_full_raw)
r2_full_raw = r2_score(yf_test, preds_full_raw)
tau_full, tau_full_p = kendalltau(yf_test, preds_full_raw)

print(f"MAPE (log scale):  {mape_full_log:.4f}")
print(f"R2   (log scale):  {r2_full_log:.4f}")
print(f"MAPE (raw, expm1): {mape_full_raw:.4f}")
print(f"R2   (raw, expm1): {r2_full_raw:.4f}")
print(f"Kendall-Tau:       {tau_full:.4f}  (p={tau_full_p:.2e})")
print()

test_family_full = df2.loc[Xf_test.index, "family"]
for fam, label in [("MLP", "BUTTER-E"), ("CNN", "EC-NAS")]:
    mask = (test_family_full == fam).values
    tau_f, tau_f_p = kendalltau(yf_test[mask], preds_full_raw[mask])
    print(f"  scored on {label:10s} subset (n={mask.sum():5d}):  Kendall-Tau={tau_f:.4f}")

print()
print("feature importances:")
print(pd.Series(rf_full.feature_importances_, index=FEATURES_FULL).sort_values(ascending=False))

**Result** (executed once already; re-run in VS Code to attach outputs):

| | MAPE (raw) | R² (raw) | Kendall-Tau |
|---|---:|---:|---:|
| Pooled, 5 core features only (before these fixes) | 1.252 | 0.132 | 0.338 |
| **Pooled, full 19-feature set (this run)** | **0.089** | **0.971** | **0.938** |

**A dramatic improvement, consistent with everything found since.** MAPE drops from 125% to 8.9%; R² from 0.13 to 0.97; Kendall-Tau from 0.34 to 0.94. Split by family on this same pooled test set: BUTTER-E tau=0.936, EC-NAS tau=0.860 — both essentially match their own solo within-family results (`03a`: 0.936; `03b`: 0.861) almost exactly, confirming pooling with masked auxiliary features still isn't causing cross-family interference, same conclusion as the earlier pooled-Kendall-Tau diagnostic, just at a much higher performance level now.

`n_observations` dominates feature importance (~0.60+), consistent with `03a`/`03h` — dataset size, not task type or class structure, is still doing almost all the work. `epochs` now has small but nonzero importance (unlike within BUTTER-E alone, where it was exactly 0) — expected, since `epochs` is 3000 for every BUTTER-E row and 4 for every EC-NAS row here, so at the pooled level it doubles as a (redundant, since `family` isn't itself a feature) family-membership signal.

This is now the strongest result in the whole project so far and the natural candidate feature set to carry into the RQ1–4 experimental matrix — though the family-holdout tests (RQ1) still need to be run properly (this notebook only uses a standard random split, which mixes both families into both train and test) before treating this number as representative of cross-family generalization.

### Pooled re-run with measurement-bias-corrected EC-NAS energy (RQ3)

`02d_measurement_correction.ipynb` applies a `×1.25` correction to EC-NAS's energy (midpoint of Fischer et al. 2025's 20-30% Carbontracker-vs-hardware-watt-meter underestimation range — see that notebook for the caveat on this being an approximation, plus a flagged inconsistency against `01a`'s own earlier "up to 40%" citation of the same source, not resolved there). Same RF, log1p target, same `random_state=42` split, same 20-feature set as the section above — only the EC-NAS portion of `target` changes (`target_corrected`, BUTTER-E rows unaffected).

In [ ]:
# BUILD CORRECTED TARGET — bring target_corrected in from ec_nas_features.csv via run_id;
# BUTTER-E rows have no correction, so they keep their original target unchanged.

ec_nas_corrected = pd.read_csv("../../data/processed/ec_nas/ec_nas_features.csv")
corrected_map = ec_nas_corrected.set_index("run_id")["target_corrected"]

df2["target_corrected"] = df2["run_id"].map(corrected_map)
df2["target_final"] = df2["target_corrected"].fillna(df2["target"])

print("rows with a correction applied:", (df2["target_final"] != df2["target"]).sum())
print("EC-NAS row count (should match):", (df2["family"] == "CNN").sum())

In [ ]:
# TRAIN + EVALUATE — RF, log1p(target_final), same split, same 20-feature set

X_corr = df2[FEATURES_FULL]
y_corr = df2["target_final"]
y_corr_log = np.log1p(y_corr)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_corr, y_corr, test_size=0.2, random_state=42)
_, _, yc_train_log, yc_test_log = train_test_split(X_corr, y_corr_log, test_size=0.2, random_state=42)

rf_corr = random_forest.build_model()
rf_corr.fit(Xc_train, yc_train_log)
preds_corr_raw = np.expm1(rf_corr.predict(Xc_test))

mape_corr = mean_absolute_percentage_error(yc_test, preds_corr_raw)
r2_corr = r2_score(yc_test, preds_corr_raw)
tau_corr, tau_corr_p = kendalltau(yc_test, preds_corr_raw)

print(f"MAPE={mape_corr:.4f}  R2={r2_corr:.4f}  Kendall-Tau={tau_corr:.4f}  (p={tau_corr_p:.2e})")

test_family_corr = df2.loc[Xc_test.index, "family"]
for fam, label in [("MLP", "BUTTER-E"), ("CNN", "EC-NAS")]:
    mask = (test_family_corr == fam).values
    r2_f = r2_score(yc_test[mask], preds_corr_raw[mask])
    tau_f, _ = kendalltau(yc_test[mask], preds_corr_raw[mask])
    print(f"  scored on {label:10s} subset (n={mask.sum():5d}):  R2={r2_f:.4f}  Kendall-Tau={tau_f:.4f}")

**Result** (executed once already; re-run in VS Code to attach outputs):

| | MAPE (raw) | R² (raw) | Kendall-Tau |
|---|---:|---:|---:|
| Pooled, uncorrected EC-NAS | 0.089 | 0.971 | 0.938 |
| **Pooled, corrected EC-NAS (×1.25)** | **0.0892** | **0.9713** | **0.9381** |

**Virtually no change — and there's a concrete reason for that, not just "the correction didn't matter."** Scored by family on this same test set: BUTTER-E R²=0.971/Tau=0.936 (both unaffected, as expected — no correction was applied there), EC-NAS R²=0.964/Tau=0.861 (also essentially unchanged from the uncorrected run's 0.958/0.861 numbers reported in `03b`).

**Why a uniform 25% level shift on one entire sub-population barely moves fit-quality metrics:** the model can already identify "this row is EC-NAS" near-perfectly from other features (`n_observations`, `task_encoded`, `is_gpu` are all exactly 0 only for EC-NAS rows, by construction of the masking scheme). A Random Forest that already isolates EC-NAS rows into their own leaves can simply refit those leaves' output values to the new, uniformly-shifted target — the *relationship* it's fitting within EC-NAS doesn't change, only the absolute level of what it's predicting there. MAPE and R² are computed relative to the (also corrected) target, so a clean uniform shift that the model can fully absorb shows up as no change in fit quality, by construction — this is the expected result for a correction of this specific form (single scalar multiplier applied to an entire, already-separable sub-population), not evidence that the correction is irrelevant.

**What the correction actually changes is not "how well the model fits," but what real-world energy value its EC-NAS-region predictions correspond to** — every prediction for an EC-NAS-like input is now ~25% higher in absolute joules than before, which matters for anything downstream that reports or compares absolute energy figures (e.g. a cross-dataset energy-savings estimate), even though it's invisible to relative fit metrics like these. Given that, and given the flagged 20-30% vs. "up to 40%" inconsistency in `02d`, resolving the correct figure from Fischer et al. (2025) directly matters more for reporting believable absolute energy numbers than for anything measured in this notebook.